# IIC-3641 GML UC

## Actividad en clase

Vamos a usar GAT para mirar **a quién le pone atención** una red de atención.

Trabajaremos con el dataset **`MinesweeperDataset`** de dgl: un tablero de Buscaminas de 100 × 100 donde cada nodo es una celda conectada a sus 8 vecinas, y el 20% de las celdas tiene una mina. Las features son un one-hot de 7 posiciones: la primera indica que la celda está **oculta**, y las otras seis que está destapada y muestra **0, 1, 2, 3, 4 o 5+** minas en su vecindario. Ojo con dos cosas: la mitad del tablero está oculta, y el número de una celda destapada cuenta las minas **de sus vecinas**, así que se muestra aunque ella misma sea una mina.

La tarea es predecir si una celda **es una mina**. Como solo el 20% lo es, evalúe con **AUC** y no con accuracy.

- Lea el dataset y muestre que la feature de una celda **no dice casi nada** sobre si ella es una mina: calcule la probabilidad de ser mina para cada uno de los 7 tipos de celda, y entrene además un MLP que no use el grafo.
- Entrene una **GAT** y compárela con dos controles: un **GCN** y **la misma GAT con la atención congelada en uniforme** ($\alpha_{ij} = 1/d_i$). Use las particiones que trae el dataset y reporte el AUC medio sobre tres de ellas.
- **Obtenga los coeficientes de atención** (`get_attention=True` en `GATConv`) y mida a qué **tipo de vecina** le pone atención el modelo, comparando contra el reparto uniforme $1/d_i$. ¿Le baja el peso a las celdas ocultas? ¿Cuál es la vecina que más mira?
- En Buscaminas hay una regla infalible: **si una celda destapada muestra 0, ninguna de sus 8 vecinas es mina.** Verifique si la regla se cumple en los datos y si la GAT la aprendió.
- Explique por qué, si la atención distingue tan bien a las vecinas inútiles, la ventaja en AUC sobre la atención uniforme es **menor a un punto**. Piense en qué le aporta al promedio una celda oculta: ¿es información engañosa o simplemente constante?
- Cuanto termine, me avisa para entregarle una **L (logrado)**.
- Recuerde que las L otorgan un bono en la nota final de la asignatura.

***Tiene hasta el final de la clase.***

In [1]:
import os

os.environ["DGLBACKEND"] = "pytorch"
import warnings

import dgl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from dgl.nn.pytorch import GATConv, GraphConv
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.__version__, dgl.__version__, device)

2.7.1+cu118 2.4.0+cu118 cuda


In [2]:
g = dgl.data.MinesweeperDataset()[0]
g = dgl.add_self_loop(dgl.remove_self_loop(g)).to(device)

X, Y = g.ndata["feat"], g.ndata["label"]
src, dst = g.edges()
LADO = 100
TIPOS = ["oculta", "0 minas", "1 mina", "2 minas", "3 minas", "4 minas", "5+ minas"]
tipo = X.argmax(1)          # 0 = oculta, 1..6 = cantidad de minas vecinas
grado = g.in_degrees()      # incluye el self-loop

print(g)
print(f"Minas: {Y.float().mean():.1%}   |   celdas ocultas: {(tipo == 0).float().mean():.1%}")

Done loading data from cached files.
Graph(num_nodes=10000, num_edges=88804,
      ndata_schemes={'feat': Scheme(shape=(7,), dtype=torch.float32), 'label': Scheme(shape=(), dtype=torch.int64), 'train_mask': Scheme(shape=(10,), dtype=torch.bool), 'val_mask': Scheme(shape=(10,), dtype=torch.bool), 'test_mask': Scheme(shape=(10,), dtype=torch.bool)}
      edata_schemes={})
Minas: 20.0%   |   celdas ocultas: 50.0%


### Continúe...